# Background

- **Author**: Codex
- **Created At**: 2026-10-13
- **Research Motivation and Context (why are we interested in the findings?)：** 這次用假訂單資料練習基本的敘述統計，先看每筆訂單買了幾件、花了多少錢，以及金額的分布。資料讀自 `data/processed/demo_orders_clean.csv`，共 115 筆，清理方式見 [整理後資料說明](../data/processed/demo_orders_documentation.ipynb)。這裡使用全部整理後的資料。
- **Main Findings and Takeaways：** 115 筆訂單的總金額為 62,071 元，平均每筆 539.75 元，中位數為 485 元。平均每筆購買約 3.01 件商品。三個商品類別的訂單數接近，A、C 各有 39 筆，B 有 37 筆。
- **Future Direction：** 下一步可以畫出訂單金額分布，看看平均值和中位數的差距來自哪些金額區間。

資料是人工生成的，以下結果只用來練習分析與表格呈現。這份 notebook 已執行完畢，表格保留本次執行結果。

第一次執行時，先用 `python data/code/demo_generate_orders.py` 產生原始資料，再執行 [處理 notebook](../data/code/demo_orders_processing.ipynb)。在有 pandas 的 Python Jupyter 環境即可執行本文件。

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
from IPython.display import display

# 從 notebook 所在位置往上找專案，從根目錄開啟也能執行。
folder = Path.cwd().resolve()
for project_root in [folder, *folder.parents]:
    if (project_root / "AGENTS.md").is_file() and (project_root / "templates").is_dir():
        break
else:
    raise FileNotFoundError("找不到專案根目錄，請從專案內開啟 notebook。")

In [2]:
data_path = (project_root / "data/processed/demo_orders_clean.csv").resolve()
expected_hash = "566d03f1701b16128fd119cc2776d775787816fb13f3d7e6d799988f6ce43104"

if data_path.parent != project_root / "data/processed":
    raise ValueError("資料路徑超出預期目錄。")
if not data_path.is_file():
    raise FileNotFoundError(f"找不到資料：{data_path.name}，請先產生資料。")

# 確認讀到的是這份文件記錄的資料版本。
file_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
if file_hash != expected_hash:
    raise ValueError("資料版本已改變，請先核對來源與說明文件。")

df = pd.read_csv(data_path)
expected_columns = [
    "order_id", "order_date", "category", "quantity", "unit_price_twd",
    "order_amount_twd",
]
assert list(df.columns) == expected_columns
assert len(df) == 115 and df["order_id"].is_unique
assert not df.isna().any().any()
assert set(df["category"]) == {"A", "B", "C"}
assert (df["order_amount_twd"] == df["quantity"] * df["unit_price_twd"]).all()

print(f"分析樣本前 10 筆（總 N={len(df)}，沒有再排除資料）")
display(df.head(10))

分析樣本前 10 筆（總 N=115，沒有再排除資料）


,order_id,order_date,category,quantity,unit_price_twd,order_amount_twd
0,D0001,2026-09-01,A,1,80,80
1,D0002,2026-09-02,B,3,97,291
2,D0003,2026-09-03,C,5,114,570
3,D0004,2026-09-04,A,2,131,262
4,D0005,2026-09-05,B,4,148,592
5,D0006,2026-09-06,C,1,165,165
6,D0007,2026-09-07,A,3,182,546
7,D0008,2026-09-08,B,5,199,995
8,D0010,2026-09-10,A,4,233,932
9,D0011,2026-09-11,B,1,250,250


In [3]:
# 訂單編號和日期用來辨識資料，這裡整理件數、單價、金額及商品類別。
variable_names = {
    "quantity": "購買件數（件）",
    "unit_price_twd": "每件單價（新臺幣元）",
    "order_amount_twd": "訂單金額（新臺幣元）",
}
stat_names = {
    "count": "有效 N",
    "mean": "平均",
    "std": "標準差",
    "min": "最小值",
    "25%": "第25百分位",
    "50%": "中位數",
    "75%": "第75百分位",
    "max": "最大值",
    "missing_n": "缺值 N",
}

numeric_data = df[list(variable_names)]
summary = numeric_data.describe(percentiles=[0.25, 0.5, 0.75]).T
summary["missing_n"] = numeric_data.isna().sum()
summary = summary.rename(index=variable_names, columns=stat_names)

print(f"表 1：分析變數敘述統計（N={len(df)}；有效 N 為非缺值筆數）")
display(summary.round(2))

category_counts = df["category"].value_counts().sort_index()
category_table = category_counts.to_frame("筆數")
category_table.index.name = "商品類別"
category_table["百分比"] = category_counts / df["category"].notna().sum() * 100

print(f"表 2：商品類別分布（N={len(df)}）")
print(f"百分比以全部非缺值訂單為分母；類別缺值 {df['category'].isna().sum()} 筆。")
display(category_table.round(2))

表 1：分析變數敘述統計（N=115；有效 N 為非缺值筆數）


,有效 N,平均,標準差,最小值,第25百分位,中位數,第75百分位,最大值,缺值 N
購買件數（件）,115.0,3.01,1.44,1.0,2.0,3.0,4.0,5.0,0
每件單價（新臺幣元）,115.0,179.19,64.83,80.0,122.5,182.0,233.0,284.0,0
訂單金額（新臺幣元）,115.0,539.75,337.45,80.0,262.0,485.0,750.0,1420.0,0


表 2：商品類別分布（N=115）
百分比以全部非缺值訂單為分母；類別缺值 0 筆。


,筆數,百分比
商品類別,,
A,39,33.91
B,37,32.17
C,39,33.91


### The actual analysis starts below

上面的表格整理了購買件數、單價、訂單金額與商品類別。接著計算全部訂單的總金額及平均金額，並將表 1 存成 `finding/demo_orders_summary.csv`，方便之後查看。

In [4]:
total_amount = df["order_amount_twd"].sum()
average_amount = df["order_amount_twd"].mean()
print(f"{len(df)} 筆訂單的總金額為 {total_amount:,} 元，平均每筆 {average_amount:,.2f} 元。")

summary_export = summary.rename_axis("變數").reset_index()
output_path = (project_root / "finding/demo_orders_summary.csv").resolve()
if output_path.parent != project_root / "finding":
    raise ValueError("輸出路徑超出預期目錄。")

# 重跑時可接受相同結果，但不蓋掉內容不同的既有檔案。
csv_bytes = summary_export.to_csv(index=False, lineterminator="\n").encode("utf-8")
if output_path.exists():
    if output_path.read_bytes() != csv_bytes:
        raise FileExistsError("輸出檔已存在且內容不同，請先確認要保留的版本。")
else:
    with output_path.open("xb") as file:
        file.write(csv_bytes)

saved_summary = pd.read_csv(output_path)
pd.testing.assert_frame_equal(summary_export, saved_summary, check_exact=False, rtol=1e-12)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == expected_hash

print(f"摘要表已存至 finding/{output_path.name}，並重新讀取核對。")
print(f"SHA-256：{hashlib.sha256(output_path.read_bytes()).hexdigest()}")

115 筆訂單的總金額為 62,071 元，平均每筆 539.75 元。
摘要表已存至 finding/demo_orders_summary.csv，並重新讀取核對。
SHA-256：a8868e7b18cdbf7cd2b95f73c650c63e20a655261fecdbfac8ff3e2179e8ab8c


平均訂單金額為 539.75 元，比中位數 485 元高約 54.75 元，表示金額較高的訂單把平均值往上拉。金額的標準差約為 337.45 元，最小值為 80 元，最大值為 1,420 元，訂單之間仍有明顯差異。

A、B、C 三類的訂單占比分別約為 33.91%、32.17%、33.91%。這裡只是描述這份資料的組成，沒有比較各類別的消費差異，也沒有做因果推論。